# Generate meta cells 

This work was done by Gabija

In [ ]:
#!pip install -q SEACells scanpy pyhere

In [1]:
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
import SEACells
from joblib import Parallel, delayed

sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi

/work/islet_cartography_scrna/scrna_cartography_gwas/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mi.create_directories(dir_path = str(here('data/meta_cell/files')))
mi.create_directories(dir_path = str(here('data/meta_cell/plots')))
mi.create_directories(dir_path = str(here('data/meta_cell/objects')))

/work/islet_cartography_scrna/data/meta_cell/files Directory already exists!
/work/islet_cartography_scrna/data/meta_cell/plots Directory already exists!
/work/islet_cartography_scrna/data/meta_cell/objects Directory already exists!


In [3]:
# Paths
base_dir = str(here('data/meta_cell/'))
plot_dir = os.path.join(base_dir, 'plots') 
files_dir = os.path.join(base_dir, 'files') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

# Data Loading

In [6]:
ad = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"), backed='r')

In [ ]:
sc.pl.scatter(ad, basis='umap', color='manual_annotation', frameon=False)

# Running SeaCells

In [ ]:
print(f'Num of metacells: {len(ad.obs) / 75}')

In [ ]:
donor_metacell_adatas = []
CELLS_PER_METACELL = 50 
MIN_CELLS = 30 

all_donor_ids = list(ad.obs["ic_id_donor_overall"].unique())
total_donors = len(all_donor_ids)
#plot_subset = random.sample(all_donor_ids, k=min(3, total_donors))

print(f"Starting processing for {total_donors} donors...")

for i, donor_id in enumerate(all_donor_ids):
    ad_d = ad[ad.obs["ic_id_donor_overall"] == donor_id].to_memory()

    n_cells = ad_d.n_obs
    if n_cells < MIN_CELLS:
        print(f"[{i+1}/{total_donors}] Skipping {donor_id}: {n_cells} cells is too few.")
        continue

    print(f"[{i+1}/{total_donors}] Processing {donor_id} ({n_cells} cells)...")

    # Determine number of metacells
    # If cell count is low, we force at least 2-3 metacells
    n_mc = max(2, n_cells // CELLS_PER_METACELL)

    model = SEACells.core.SEACells(
        ad_d,
        build_kernel_on="X_latent_1", 
        n_SEACells=n_mc,
        n_waypoint_eigs=10, # Number of eigenvalues to consider when initializing metacells
        convergence_epsilon=1e-3
    )
    
    model.construct_kernel_matrix()
    model.initialize_archetypes()

    # Plot the initilization to ensure they are spread across phenotypic space
    if donor_id in all_donor_ids:
        try:
            SEACells.plot.plot_initialization(ad_d, model, save_as=os.path.join(plot_dir, f"{donor_id}_initialization.pdf"))
        except Exception as plot_err:
            print(f"Plotting failed for {donor_id}: {plot_err}")

    try:
        model.fit(min_iter=10, max_iter=100)
        mc_ad = SEACells.core.summarize_by_SEACell(
            ad_d, SEACells_label="SEACell", summarize_layer="counts"
        )

        # Metadata Transfer
        meta_cols = ['manual_annotation', 'ethnicity_broad_harmonized', 'disease_harmonized']
        for col in meta_cols:
            if col in ad_d.obs.columns:
                # Majority vote for categorical data
                counts = ad_d.obs.groupby(['SEACell', col]).size().unstack(fill_value=0)
                majority = counts.idxmax(axis=1)
                mc_ad.obs[col] = mc_ad.obs_names.map(majority)
        
        mc_ad.obs["ic_id_donor_overall"] = donor_id
        mc_ad.obs_names = [f"{donor_id}_mc_{name}" for name in mc_ad.obs_names]
        
        donor_metacell_adatas.append(mc_ad)
        
    except Exception as e:
        print(f"Model fit failed for {donor_id}: {e}")

In [ ]:
meta_ad = sc.concat(donor_metacell_adatas)
meta_ad.layers['counts'] = meta_ad.X.copy()
meta_ad.write_h5ad(os.path.join(objects_dir, "metacells_final.h5ad"))

In [ ]:
import anndata as AD
pooled_step1 = AD.concat(donor_metacell_adatas, join="outer", fill_value=0)
pooled_step1.obs_names_make_unique()
pooled_step1.write_h5ad(os.path.join(objects_dir, "metacells_per_donor.h5ad"))

metacell_counts = (
    pooled_step1.obs
    .groupby("ic_id_donor_overall")
    .size()
    .reset_index(name="n_metacells")
    .sort_values("n_metacells", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))


axes[0].bar(range(len(metacell_counts)), metacell_counts["n_metacells"], color="steelblue")
axes[0].set_xlabel("Donor (sorted by metacell count)")
axes[0].set_ylabel("Number of metacells")
axes[0].set_title(f"Metacells per donor (total: {metacell_counts['n_metacells'].sum()})")
axes[0].set_xticks([])  

sns.histplot(metacell_counts["n_metacells"], bins=30, ax=axes[1], color="steelblue")
axes[1].set_xlabel("Number of metacells")
axes[1].set_ylabel("Number of donors")
axes[1].set_title(f"Distribution (median: {metacell_counts['n_metacells'].median():.0f})")
sns.despine()

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "metacells_per_donor.pdf"))
plt.show()
plt.close()

print(metacell_counts.describe())

In [ ]:
SEACells.plot.plot_initialization(ad, model)

In [ ]:
sc.pp.normalize_total(meta_ad, target_sum=1e4)
sc.pp.log1p(meta_ad)
sc.pp.highly_variable_genes(meta_ad, n_top_genes=2000)

sc.tl.pca(meta_ad, n_comps=30)
sc.pp.neighbors(meta_ad, n_neighbors=15, use_rep="X_pca")
sc.tl.umap(meta_ad)

sc.pl.umap(meta_ad, color='manual_annotation', frameon=False, title='Metacell UMAP')

In [ ]:
table_df = meta_ad.obs.groupby('manual_annotation').size().reset_index()
table_df.columns = ['Cell Type', 'Metacells']

single_cell_counts = ad.obs['manual_annotation'].value_counts().reset_index()
single_cell_counts.columns = ['Cell Type', 'Single Cells']

df = pd.merge(table_df, single_cell_counts, on='Cell Type')
df['Ratio'] = (df['Single Cells'] / df['Metacells']).round(1)
df = df.sort_values('Single Cells', ascending=False)

fig, ax = plt.subplots(figsize=(8, len(df)*0.5 + 1)) 
ax.axis('off')
tbl = ax.table(cellText=df.values, 
               colLabels=df.columns, 
               cellLoc='center', 
               loc='center',
               edges='horizontal') 
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8) 

for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.get_text().set_weight('bold')
        cell.set_linewidth(2) 
    elif row == len(df):
        cell.set_linewidth(2) 
    else:
        cell.set_linewidth(0.5) 

plt.title('Metacell Representation by Cell Type', pad=20, weight='bold')
plt.savefig(os.path.join(plot_dir, 'metacell_summary_table.pdf'))

#### Add columns for group analysis

In [ ]:
# make a cell_type columns
meta_ad.obs['cell_type'] = meta_ad.obs['manual_annotation'].astype(str)
meta_ad.obs['cell_type_disease'] = meta_ad.obs['manual_annotation'].astype(str)

# make a celltype disease column
target_cells = list(meta_ad.obs['cell_type'].unique())

mask = meta_ad.obs['manual_annotation'].isin(target_cells)

meta_ad.obs.loc[mask, 'cell_type_disease'] = (
    meta_ad.obs.loc[mask, 'cell_type_disease'] + 
    '_' + 
    meta_ad.obs.loc[mask, 'disease_harmonized'].astype(str)
)
meta_ad.obs['cell_type_disease'] = meta_ad.obs['cell_type_disease'].astype('category')

In [8]:
meta_ad.obs['cell_type_ethnicity'] = meta_ad.obs['manual_annotation'].astype(str)
target_cells = list(meta_ad.obs['cell_type'].unique())
mask = meta_ad.obs['manual_annotation'].isin(target_cells)
meta_ad.obs.loc[mask, 'cell_type_ethnicity'] = (
    meta_ad.obs.loc[mask, 'cell_type_ethnicity'] + 
    '_' + 
    meta_ad.obs.loc[mask, 'ethnicity_broad_harmonized'].astype(str)
)
meta_ad.obs['cell_type_ethnicity'] = meta_ad.obs['cell_type_ethnicity'].astype('category')

KeyError: 'ic_id_dataset'

#### Create covariate table for scDRS analysis

In [9]:

# Creating cov for scDRS
cov_df = pd.DataFrame(index=meta_ad.obs_names)
cov_df['ic_id_dataset'] = meta_ad.obs['ic_id_dataset'].astype('category').cat.codes
cov_df['cell_nuclei'] = meta_ad.obs['cell_nuclei'].astype('category').cat.codes
cov_df.to_csv(os.path.join(files_dir, "metacell_cov.tsv"), sep = "\t")

ValueError: `shape` is inconsistent with `obs` (10459 rows instead of 9978)

In [4]:
meta_ad = sc.read_h5ad(os.path.join(objects_dir, "metacells_final.h5ad"))

#### Save 

In [ ]:
meta_ad.write_h5ad(os.path.join(objects_dir, "metacells_final.h5ad"))